# LA Studio Voice Design - Qwen3-TTS VoiceDesign 1.7B

This notebook loads exactly `qwen3-tts-1.7b-voicedesign` (`Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign`) on CUDA.
It is independent from API Gateway and refuses every other model ID.

1. Choose **Runtime -> Change runtime type -> GPU**.
2. Run all cells.
3. Copy the printed URL and token into LA Studio's Voice Design panel.


In [ ]:
!nvidia-smi
%pip install -q "qwen-tts==0.1.1" "soundfile==0.13.1" "fastapi==0.115.12" "uvicorn==0.34.3"


In [ ]:
from pathlib import Path

WORKER = Path('/content/la_studio_voice_design_worker.py')
WORKER.write_text('import torch\n\nfrom qwen_tts import Qwen3TTSModel\n\nMODEL_ID = "qwen3-tts-1.7b-voicedesign"\nMODEL_NAME = "Qwen3-TTS VoiceDesign 1.7B"\nUPSTREAM_MODEL = "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign"\nMODEL = Qwen3TTSModel.from_pretrained(\n    UPSTREAM_MODEL,\n    device_map="cuda:0",\n    dtype=torch.bfloat16,\n    attn_implementation="sdpa",\n)\n\ndef qwen_language(value: str):\n    mapping = {"auto": "Auto", "zh": "Chinese", "en": "English", "ja": "Japanese", "ko": "Korean", "de": "German", "fr": "French", "ru": "Russian", "pt": "Portuguese", "es": "Spanish", "it": "Italian"}\n    return mapping.get(value.strip().lower(), value.strip().title() or "Auto")\n\ndef design_with_exact_model(request):\n    instruction = ", ".join(part for part in (request.voice_description.strip(), request.style.strip()) if part)\n    if request.seed >= 0:\n        torch.manual_seed(request.seed)\n        torch.cuda.manual_seed_all(request.seed)\n    wavs, sample_rate = MODEL.generate_voice_design(\n        text=request.input,\n        language=qwen_language(request.language),\n        instruct=instruction,\n        temperature=request.temperature,\n    )\n    return wavs[0], sample_rate\n\nimport io\nimport os\nimport re\nimport threading\n\nimport numpy as np\nimport soundfile as sf\nimport torch\nfrom fastapi import FastAPI, Header, HTTPException\nfrom fastapi.responses import Response\nfrom pydantic import BaseModel, Field\n\nif not torch.cuda.is_available():\n    raise RuntimeError("CUDA is unavailable. In Colab choose Runtime > Change runtime type > GPU, then Run all.")\n\nTOKEN = os.environ["LA_STUDIO_COLAB_VOICE_DESIGN_TOKEN"]\nMAX_INPUT_CHARS = 4000\nMAX_OUTPUT_SECONDS = 300\nREQUEST_SLOTS = threading.BoundedSemaphore(1)\n\nclass VoiceDesignRequest(BaseModel):\n    model: str = Field(min_length=1, max_length=120)\n    input: str = Field(min_length=1, max_length=MAX_INPUT_CHARS)\n    voice_description: str = Field(min_length=1, max_length=2000)\n    style: str = Field(default="", max_length=1000)\n    language: str = Field(default="en", max_length=40)\n    temperature: float = Field(default=0.9, ge=0.1, le=2.0)\n    seed: int = Field(default=-1, ge=-1)\n\ndef authorize(authorization: str | None) -> None:\n    if authorization != "Bearer " + TOKEN:\n        raise HTTPException(status_code=401, detail="invalid worker token")\n\ndef audio_array(value):\n    if isinstance(value, (list, tuple)):\n        if not value:\n            raise RuntimeError("the selected model returned no audio")\n        value = value[0]\n    if torch.is_tensor(value):\n        value = value.detach().float().cpu().numpy()\n    audio = np.asarray(value, dtype=np.float32).reshape(-1)\n    if audio.size == 0 or not np.isfinite(audio).all():\n        raise RuntimeError("the selected model returned invalid audio")\n    return audio\n\ndef wav_response(value, sample_rate: int):\n    audio = audio_array(value)\n    if audio.size > int(sample_rate) * MAX_OUTPUT_SECONDS:\n        raise HTTPException(status_code=413, detail="generated audio exceeds the five minute output limit")\n    peak = float(np.max(np.abs(audio)))\n    if peak > 1.2:\n        audio = audio / peak\n    output = io.BytesIO()\n    sf.write(output, audio, int(sample_rate), format="WAV", subtype="PCM_16")\n    return Response(output.getvalue(), media_type="audio/wav", headers={"Cache-Control": "no-store"})\n\napp = FastAPI(title=f"LA Studio Voice Design - {MODEL_NAME}", docs_url=None, redoc_url=None, openapi_url=None)\n\n@app.get("/health")\n@app.get("/v1/health")\ndef health(authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    return {\n        "status": "ready",\n        "ready": True,\n        "device": "cuda",\n        "gpu": torch.cuda.get_device_name(0),\n        "model": MODEL_ID,\n        "upstream_model": UPSTREAM_MODEL,\n        "cpu_fallback": False,\n    }\n\n@app.get("/v1/capabilities")\ndef capabilities(authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    return {\n        "contract_version": 1,\n        "device": "cuda",\n        "capabilities": [{\n            "id": "voice-design",\n            "models": [{\n                "id": MODEL_ID,\n                "name": MODEL_NAME,\n                "upstream_model": UPSTREAM_MODEL,\n                "formats": ["wav"],\n                "device": "cuda",\n                "loaded": True,\n            }],\n        }],\n    }\n\n@app.post("/v1/audio/voice_designs")\ndef voice_design(request: VoiceDesignRequest, authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    if request.model.strip().lower() != MODEL_ID:\n        raise HTTPException(\n            status_code=409,\n            detail=f"This worker loaded \'{MODEL_ID}\', but LA Studio requested \'{request.model}\'. Open the notebook for the selected model.",\n        )\n    if not REQUEST_SLOTS.acquire(blocking=False):\n        raise HTTPException(status_code=429, detail="the Colab voice-design worker is busy; retry shortly")\n    try:\n        audio, sample_rate = design_with_exact_model(request)\n        return wav_response(audio, sample_rate)\n    except HTTPException:\n        raise\n    except Exception as error:\n        raise HTTPException(\n            status_code=503,\n            detail=f"{MODEL_NAME} voice design failed: {type(error).__name__}: {str(error)[:240]}",\n        ) from error\n    finally:\n        REQUEST_SLOTS.release()\n', encoding='utf-8')
print('Worker source:', WORKER)


In [ ]:
import json, os, re, secrets, subprocess, sys, time, urllib.error, urllib.request
from pathlib import Path

MODEL_ID = 'qwen3-tts-1.7b-voicedesign'
TOKEN = secrets.token_urlsafe(32)
STARTUP_TIMEOUT_SECONDS = 20 * 60
WORKER_LOG = Path("/content/la_studio_voice_design_worker.log")
env = os.environ.copy()
env["LA_STUDIO_COLAB_VOICE_DESIGN_TOKEN"] = TOKEN
env["PYTHONUNBUFFERED"] = "1"

def worker_log_tail() -> str:
    try:
        return WORKER_LOG.read_text(encoding="utf-8", errors="replace")[-12000:]
    except FileNotFoundError:
        return "(worker log was not created)"

def fail_startup(message: str) -> None:
    if worker.poll() is None:
        worker.terminate()
        try:
            worker.wait(timeout=10)
        except subprocess.TimeoutExpired:
            worker.kill()
    raise RuntimeError(
        message + "\\n\\n---- LA Studio worker log (last 12,000 characters) ----\\n" + worker_log_tail()
    )

with WORKER_LOG.open("w", encoding="utf-8", buffering=1) as worker_output:
    worker = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "la_studio_voice_design_worker:app", "--host", "127.0.0.1", "--port", "3924"],
        cwd="/content",
        env=env,
        stdout=worker_output,
        stderr=subprocess.STDOUT,
    )
    print("Starting exact CUDA worker; initial model download can take several minutes.")
    deadline = time.monotonic() + STARTUP_TIMEOUT_SECONDS
    last_error = "worker has not answered /health yet"
    while time.monotonic() < deadline:
        exit_code = worker.poll()
        if exit_code is not None:
            fail_startup(f"The exact-model worker exited before becoming ready (exit code {exit_code}).")
        try:
            check = urllib.request.Request(
                "http://127.0.0.1:3924/health",
                headers={"Authorization": "Bearer " + TOKEN},
            )
            with urllib.request.urlopen(check, timeout=10) as response:
                health = json.loads(response.read().decode("utf-8"))
            if (response.status == 200
                    and health.get("ready") is True
                    and health.get("device") == "cuda"
                    and health.get("model") == MODEL_ID
                    and health.get("cpu_fallback") is False):
                print("Exact CUDA worker is ready:", health)
                break
            last_error = "unexpected /health response: " + json.dumps(health, ensure_ascii=False)
        except urllib.error.HTTPError as error:
            last_error = f"/health returned HTTP {error.code}: " + error.read().decode("utf-8", errors="replace")[:1000]
        except Exception as error:
            last_error = f"/health is not ready: {type(error).__name__}: {error}"
        if int(time.monotonic()) % 30 == 0:
            print("Waiting for the exact CUDA model…", last_error)
        time.sleep(2)
    else:
        fail_startup(
            f"The exact-model worker did not become CUDA-ready within {STARTUP_TIMEOUT_SECONDS // 60} minutes. "
            f"Last health-check result: {last_error}"
        )

subprocess.run(
    ["bash", "-lc", "wget -q -O /content/cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i /content/cloudflared.deb"],
    check=True,
)
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:3924", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
public_url = None
for _ in range(120):
    line = tunnel.stdout.readline()
    print(line, end="")
    match = re.search(r"https://[^\s]+trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break
if not public_url:
    worker.terminate()
    tunnel.terminate()
    raise RuntimeError("Cloudflare tunnel URL was not found")

print("\nLA_STUDIO_COLAB_VOICE_DESIGN_URL=" + public_url)
print("LA_STUDIO_COLAB_VOICE_DESIGN_TOKEN=" + TOKEN)
print("LA_STUDIO_COLAB_VOICE_DESIGN_MODEL=" + MODEL_ID)
print("DEVICE=cuda; CPU_FALLBACK=false")
